In [1]:
-- SCV Build
CREATE or Replace table stg.SCVBase_01 as 
select	SourceSystemReference, SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId 
from	ods.scv_customer_key 
where	SourceSystemId = 1 
;
CREATE or Replace table stg.SCVBase_02 as 
select	QuoteQueryGuid, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, c.PRN as AdditionalDriverFlag
from	ods.scv_customer_key				a,
		dlk.mfq_quotequery					b,
		dlk.mfq_quotedrivers				c
where	a.SourceSystemId = 2 
and		a.SourceSystemReference = c.QuoteDriverId
and		b.QuoteQueryId = c.QuoteQueryId
Group by QuoteQueryGuid, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, c.PRN 
;
CREATE or Replace table stg.SCVBase_03 as 
select	b.QuoteCodeReference, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId,
		case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end as JointCustomerFlag 
from	ods.scv_customer_key				a,
		dlk.hfq_quotedetails				b
where	a.SourceSystemId = 3 
and		replace(a.SourceSystemReference, ':JOINT', '') = b.QuoteCodeReference
Group by b.QuoteCodeReference, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
;
CREATE or Replace table stg.SCVBase_04 as 
select	c.PolicyCode, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key				a,
		dlk.ext_home_qs_policyholderdetails	b,
		dlk.ext_home_qs_policydetails		c
where	a.SourceSystemId = 4 
and		a.SourceSystemReference = b.PolicyHolderId
and		b.PolicyId = c.PolicyId
;
CREATE or Replace table stg.SCVBase_05 as 
select	QuoteId, PolicyId, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
from	ods.scv_customer_key				a,
		dlk.ext_travel_policy				b 
where	a.SourceSystemId = 5 
and		a.SourceSystemReference = b.MapfreCustomerId
Group by QuoteId, PolicyId, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
;
CREATE or Replace table stg.SCVBase_06 as 
select	PK_ContactID, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end as JointCustomerFlag 
from	ods.scv_customer_key				a,
		dlk.life_tblcontacts				b 
where	a.SourceSystemId = 6 
and		replace(a.SourceSystemReference, ':JOINT', '') = b.PK_ContactID
Group by PK_ContactID, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end 
;
CREATE or Replace table stg.SCVBase_07 as 
select	RowKey, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key				a,
		dlk.vanquotedetails					b
where	a.SourceSystemId = 7 
and		a.SourceSystemReference = b.RowKey
Group by RowKey, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
;
CREATE or Replace table stg.SCVBase_08 as 
select	quote_number, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId , user_id as email
from	ods.scv_customer_key				a,
		dlk.ext_travel_quotes				b
where	a.SourceSystemId = 8 
and		a.SourceSystemReference = b.quote_number
Group by quote_number, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, user_id 
;
CREATE or Replace table stg.SCVBase_09 as 
select	Policy_Code PolicyCode, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key					a,
		dlk.ext_quotationstorage_policydetails	b
where	a.SourceSystemId = 9 
and		replace(a.SourceSystemReference, ':1', '') = b.Quotation_ID
Group by Policy_Code, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 10, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
CREATE or Replace table ods.SCVBase as 
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(null  as varchar(225)) as QuoteReference, cast(1 as VARCHAR(10)) AdditionalCustomerFlag  
from stg.SCVBase_01 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, QuoteQueryGuid as QuoteReference, AdditionalDriverFlag   
from stg.SCVBase_02 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, QuoteCodeReference, 1    
from stg.SCVBase_03 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, null as ClientCode, PolicyCode, SourceSystemReference as QuoteReference, 1    
from stg.SCVBase_04 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode, CAST(PolicyId as varchar(225)) as PolicyCode, CAST(QuoteId as varchar(225)) as QuoteReference, CAST(1 AS VARCHAR(10))   
from stg.SCVBase_05 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(PK_ContactID as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(null  as varchar(225)) as QuoteReference, CAST(1 + JointCustomerFlag AS VARCHAR(10))   
from stg.SCVBase_06 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, RowKey as QuoteReference, CAST(1 AS VARCHAR(10))  
from stg.SCVBase_07 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, email as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(quote_number as varchar(225)) as QuoteReference, CAST(1 AS VARCHAR(10)) 
from stg.SCVBase_08 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, left(PolicyCode,6) as ClientCode, cast(PolicyCode as varchar(225)), left(SourceSystemReference, 36) as QuoteReference, CAST(1 AS VARCHAR(10))  
from stg.SCVBase_09
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
Create or Replace Table stg.RelayCustomers as 
select	min(SCV_Customer_Key) SCV_Customer_Key,  ClientCode
from	ods.scvbase			a 
Where	a.SourceSystemId in (1,4,9) 
and		len(a.ClientCode) = 6
Group by ClientCode
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
Create or Replace Table stg.BOCustomers as 
select	min(SCV_Customer_Key) SCV_Customer_Key,  PolicyCode
from	ods.scvbase			a 
Where	a.SourceSystemId in (1,4,9) 
and ClientCode is null 
Group by PolicyCode
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
Create or Replace Table ods.ScvEadmLink as 
select	case 
                when b.SCV_Customer_Key < 10000000000000 then b.SCV_Customer_Key
                when c.SCV_Customer_Key < 10000000000000 then c.SCV_Customer_Key
                else a.SCV_Customer_Key 
        end as SCV_Customer_Key_Final,
        a.*
from	ods.scvbase			a 
left join 
        stg.RelayCustomers  b 
            on  b.ClientCode = left(a.PolicyCode,6)
            and	a.SourceSystemId in (1,4,9)
            and	len(a.PolicyCode) <= 15
left join 
        stg.BOCustomers         c 
        on  c.PolicyCode = a.PolicyCode 
            and	a.SourceSystemId in (1,4,9)
            and	len(a.PolicyCode) <= 15
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
-- does this pass 
select scvcounts   , count(*) 
from (	
		select  PolicyCode, count(distinct SCV_Customer_Key_Final) scvcounts   
        from    ods.scveadmlink 
		Where   AdditionalCustomerFlag <= 1 
        and     PolicyCode is not null 
		Group by PolicyCode
		
		) x Group by scvcounts    order by 1  
;

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 18, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [11]:
create or replace table ods.scveadmlinkfinal as 
select  SCV_Customer_Key_Final as SCV_Customer_Key,
        ChillSourceCustomerKey,
        ChillSourceAddressKey, 
        SourceSystemId, 
        SourceSystemReference, 
        ClientCode,
        PolicyCode, 
        QuoteReference, 
        AdditionalCustomerFlag  
from    ods.scveadmlink
Where   AdditionalCustomerFlag <= 1

StatementMeta(, 3d699288-d374-4d18-a321-23687b3fc52e, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
Create or Replace table stg.episodeeventstream as 
select	ROW_Number() over (partition by 1 Order by PolicyTypeGroup, SourceQuoteReference, SourcePolicyReference, SourceCustomerReference) EventId	,
		EventSourceReference	,
		SourceQuoteReference	,
		SourcePolicyReference	,
		SourceCustomerReference	,
		SourceSystemId	,
		PolicyTypeGroup	,
		EventDateTime	,
		EventDate	,
		EventTypeId	,
		EventDescription,	
		Grain
from	ods.episodeeventstream 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
Create or Replace table ods.scvepisodeeventstream as 
select	EventId,
		EventSourceReference	,
		SourceQuoteReference	,
		SourcePolicyReference	,
		SourceCustomerReference	,
		SourceSystemId	,
		PolicyTypeGroup	,
		EventDateTime	,
		EventDate	,
		EventTypeId	,
		EventDescription	,
		Grain,
        SCV_Customer_Key  as SCV_Customer_Key
from	stg.episodeeventstream a, (select ChillSourcePolicyReference, SCV_Customer_Key from ods.scveadmlinkfinal Group by  ChillSourcePolicyReference, SCV_Customer_Key) b 
Where   a.SourcePolicyReference = b.ChillSourcePolicyReference 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		a.SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key
from	stg.episodeeventstream a 
join 
        (select ChillSourceCustomerReference, SCV_Customer_Key from ods.scveadmlinkfinal Group by  ChillSourceCustomerReference, SCV_Customer_Key) b 
            on a.SourceCustomerReference = b.ChillSourceCustomerReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		b.SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key
from    stg.episodeeventstream  a 
join
        ods.scveadmlinkfinal    b 
            on a.SourceQuoteReference = b.ChillSourceQuoteReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
and     a.Grain = 'Quote' 
and		a.PolicyTypeGroup ='Travel' 
and     b.SourceSystemId = 5 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		b.SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key
from    stg.episodeeventstream  a 
join
        ods.scveadmlinkfinal    b 
            on a.SourceQuoteReference = b.ChillSourceQuoteReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
and     a.Grain = 'Quote' 
and		a.PolicyTypeGroup = 'Motor' 
and		b.SourceSystemId = 2 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		b.SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key
from    stg.episodeeventstream  a 
join
        ods.scveadmlinkfinal    b 
            on a.SourceQuoteReference = b.ChillSourceQuoteReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
and     a.Grain = 'Quote' 
and		a.PolicyTypeGroup = 'Home' 
and		b.SourceSystemId = 3 
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		Null SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ConversationId,
                SCV_Customer_Key
        from    ods.customercallevents  
        Group by ConversationId,
                SCV_Customer_Key)   b 
            on a.EventSourceReference = b.ConversationId
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
and     a.Grain = 'Call' 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		Null SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ConversationId,
                SCV_Customer_Key
        from    ods.customercallevents  
        Group by ConversationId,
                SCV_Customer_Key)   b 
            on a.EventSourceReference = b.ConversationId
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		Null SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ConversationId,
                SCV_Customer_Key
        from    ods.customercallevents  
        Group by ConversationId,
                SCV_Customer_Key)   b 
            on a.SourcePolicyReference = b.ConversationId
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
        a.EventSourceReference	,
        a.SourceQuoteReference	,
        a.SourcePolicyReference	,
        a.SourceCustomerReference	,
        b.SourceSystemId	,
        a.PolicyTypeGroup	,
        a.EventDateTime	,
        a.EventDate	,
        a.EventTypeId	,
        a.EventDescription	,
        a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ChillSourceQuoteReference,
                SCV_Customer_Key,
                SourceSystemId
        from    ods.scveadmlinkfinal
        Group by ChillSourceQuoteReference,
                SCV_Customer_Key,
                SourceSystemId)   b 
            on a.SourcePolicyReference = b.ChillSourceQuoteReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
        a.EventSourceReference	,
        a.SourceQuoteReference	,
        a.SourcePolicyReference	,
        a.SourceCustomerReference	,
        b.SourceSystemId	,
        a.PolicyTypeGroup	,
        a.EventDateTime	,
        a.EventDate	,
        a.EventTypeId	,
        a.EventDescription	,
        a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ChillSourceQuoteReference,
                SCV_Customer_Key,
                SourceSystemId
        from    ods.scveadmlinkfinal
        Group by ChillSourceQuoteReference,
                SCV_Customer_Key,
                SourceSystemId)   b 
            on a.SourceQuoteReference = b.ChillSourceQuoteReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
        a.EventSourceReference	,
        a.SourceQuoteReference	,
        a.SourcePolicyReference	,
        a.SourceCustomerReference	,
        1 SourceSystemId	,
        a.PolicyTypeGroup	,
        a.EventDateTime	,
        a.EventDate	,
        a.EventTypeId	,
        a.EventDescription	,
        a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        (Select ChillSourceCustomerReference,
                SCV_Customer_Key
        from    ods.scveadmlinkfinal
        Where   SourceSystemId = 1 
        Group by ChillSourceCustomerReference,
                SCV_Customer_Key
                )   b 
            on left(a.SourcePolicyReference,6) = b.ChillSourceCustomerReference
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
create or replace table ods.MissingCustomers as 
Select  -1 * Row_Number() over (Partition by 1 Order by a.EventTypeId) as SCV_Customer_Key,
        a.EventId,
        a.EventTypeId,
        a.EventSourceReference,
        a.SourceCustomerReference,
        a.SourceQuoteReference,
        a.SourcePolicyReference
from    stg.episodeeventstream a 
left join 
        ods.scvepisodeeventstream b 
            on a.EventId = b.EventId 
where   b.EventId is null          
;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
Insert into ods.scvepisodeeventstream  
(
    EventId,
    EventSourceReference,
    SourceQuoteReference,
    SourcePolicyReference,
    SourceCustomerReference,
    SourceSystemId,
    PolicyTypeGroup,
    EventDateTime,
    EventDate,
    EventTypeId,
    EventDescription,
    Grain,
    SCV_Customer_Key
)
select	a.EventId,
		a.EventSourceReference	,
		a.SourceQuoteReference	,
		a.SourcePolicyReference	,
		a.SourceCustomerReference	,
		0 SourceSystemId	,
		a.PolicyTypeGroup	,
		a.EventDateTime	,
		a.EventDate	,
		a.EventTypeId	,
		a.EventDescription	,
		a.Grain,
        b.SCV_Customer_Key 
from    stg.episodeeventstream      a
join
        ods.MissingCustomers        b 
            on a.EventId = b.EventId 
left JOIN
         ods.scvepisodeeventstream  c     
            on a.EventId = c.EventId
Where   c.EventId is null 
;


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
Create or Replace Table ods.scvepisodeeventstreamfinal as 
select  EventId,
        EventSourceReference,
        SourceQuoteReference,
        SourcePolicyReference,
        SourceCustomerReference,
        SourceSystemId,
        PolicyTypeGroup,
        EventDateTime,
        EventDate,
        EventTypeId,
        EventDescription,
        Grain,
        SCV_Customer_Key
From    (   select  EventId,
                    EventSourceReference,
                    SourceQuoteReference,
                    SourcePolicyReference,
                    SourceCustomerReference,
                    SourceSystemId,
                    PolicyTypeGroup,
                    EventDateTime,
                    EventDate,
                    EventTypeId,
                    EventDescription,
                    Grain,
                    SCV_Customer_Key,
                    Row_number() over (partition by EventId Order by SCV_Customer_Key) as rno 
            from    ods.scvepisodeeventstream  
            ) x 
Where   rno = 1             

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
select Grain, count(*) from ods.scvepisodeeventstreamfinal  Group by  Grain Order by 1 

StatementMeta(, ccec562e-9346-426b-82c0-27ddbb643b90, 18, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 2 fields>

In [16]:
select Grain, count(*) from ods.scvepisodeeventstreamfinal  Group by Grain Order by 1 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

In [17]:
-- DQ 1
select a.Grain, count(*) from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId where b.EventId is null Group by  a.Grain Order by 1 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [18]:
-- DQ 2
select counts, count(*) from (select EventId, count(*) counts from  ods.scvepisodeeventstreamfinal Group by EventId having count(*) > 1 ) x Group by counts order by 1 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [12]:
create or replace table ods.MissingCustomers as 
Select  -1 * Row_Number() over (Partition by 1 Order by a.EventTypeId) as SCV_Customer_Key,
        a.EventId,
        a.EventTypeId,
        a.EventSourceReference,
        a.SourceCustomerReference,
        a.SourceQuoteReference,
        a.SourcePolicyReference
from    stg.episodeeventstream a 
left join 
        ods.scvepisodeeventstreamfinal b 
            on a.EventId = b.EventId 
where   b.EventId is null          
;

StatementMeta(, ccec562e-9346-426b-82c0-27ddbb643b90, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
select * from ods.scvepisodeeventstreamfinal where EventId = 39460983

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 13 fields>

In [20]:
select  *from ods.MissingCustomers 
Where EventTypeId = 'A1' 


StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 7 fields>

In [13]:
select EventTypeId, count(*) from ods.MissingCustomers
Group by EventTypeId 
Order by 1 ;

StatementMeta(, ccec562e-9346-426b-82c0-27ddbb643b90, 14, Finished, Available, Finished, False)

<Spark SQL result set with 53 rows and 2 fields>

In [21]:
select EventTypeId, count(*) from ods.MissingCustomers
Group by EventTypeId 
Order by 1 ;

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 22, Finished, Available, Finished, False)

<Spark SQL result set with 98 rows and 2 fields>

In [22]:
select a.EventTypeId, count(*) from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
where b.EventId is null  
Group by a.EventTypeId
limit 100 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [23]:
select a.Grain, count(*) from stg.episodeeventstream a join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
where  a.EventTypeId = 'R3b.F1'
Group by a.Grain
Order by 2 desc 
limit 100 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 24, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [24]:
select a.Grain, count(*) from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
where b.EventId is null  and a.EventTypeId = 'R3b.F1'
Group by a.Grain
Order by 2 desc 
limit 100 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [25]:
    select a.* from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
    where b.EventId is null  
    and a.Grain = 'Customer' and a.EventTypeId not in ('HA1.E2','HA1.E1','A3.E1') -- all seem to be CRM 22/09/2026
    limit 100 
    /*
    Call
Customer
Payment Attempt
Policy
Quote
*/

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 12 fields>

In [26]:
select a.* from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
where b.EventId is null  
and a.Grain = 'Policy' and a.EventTypeId not in ('A5.1')
limit 100 

StatementMeta(, d9923fdd-1eb2-48ca-826f-db8e3c559c7e, 27, Submitted, Running, Running, True)

In [ ]:
select a.* from stg.episodeeventstream a left join ods.scvepisodeeventstreamfinal b on a.EventId = b.EventId 
where b.EventId is null  
and a.Grain = 'Quote' and a.EventTypeId not in ('HA2.E1', 'A0')
limit 100 

StatementMeta(, , -1, Waiting, , Waiting, True)